# Real Data Cleaning and Transformation Checks

This notebook **verifies** the real v1 cleaned and curated datasets. The runnable raw-to-curated transformation lives in `07_production_etl`.

In [ ]:
from pyspark.sql import functions as F

storage = "stflight60106354"
processed_path = (
    f"abfss://processed@{storage}.dfs.core.windows.net/"
    "flight_delays/2025/v1/clean_parquet/"
)
curated_path = (
    f"abfss://curated@{storage}.dfs.core.windows.net/"
    "flight_delays/2025/v1/model_ready/"
)

clean = spark.read.parquet(processed_path)
print("PASS: Real processed data loaded")
print("Columns:", len(clean.columns))
print("Rows:", clean.count())
display(clean.limit(10))


In [ ]:
# Validate the real cleaning and derived columns.
required = [
    "FlightDate", "Year", "Month", "Reporting_Airline", "Origin",
    "Dest", "CRSDepTime", "ArrDelay", "ArrDel15", "DepHour",
    "DayOfWeek", "IsWeekend", "Quarter", "Route"
]
missing = [c for c in required if c not in clean.columns]
assert not missing, f"Missing processed columns: {missing}"

checks = clean.agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("FlightDate").isNull(), 1).otherwise(0)).alias("missing_dates"),
    F.sum(F.when(~F.col("ArrDel15").isin(0, 1) | F.col("ArrDel15").isNull(), 1).otherwise(0)).alias("bad_targets"),
    F.sum(F.when(~F.col("DepHour").between(0, 23) | F.col("DepHour").isNull(), 1).otherwise(0)).alias("bad_hours"),
    F.sum(F.when(F.col("ArrDel15") != F.when(F.col("ArrDelay") >= 15, 1).otherwise(0), 1).otherwise(0)).alias("target_mismatches")
).first()

assert checks["total"] == 6879484, f"Unexpected row count: {checks['total']}"
for field in ["missing_dates", "bad_targets", "bad_hours", "target_mismatches"]:
    assert checks[field] == 0, f"Validation failed: {field} = {checks[field]}"

print("PASS: Real cleaned data validation")
print("Validated rows:", checks["total"])
print("Missing dates / bad targets / bad hours / target mismatches: 0")


In [ ]:
# Read the model-ready result and check that post-flight values are excluded.
model_ready = spark.read.parquet(curated_path)

expected_model_columns = [
    "FlightDate", "Year", "Month", "Reporting_Airline", "Origin",
    "Dest", "CRSDepTime", "DepHour", "DayOfWeek", "IsWeekend",
    "Quarter", "Route", "ArrDel15"
]
assert set(model_ready.columns) == set(expected_model_columns)
assert "ArrDelay" not in model_ready.columns
assert model_ready.count() == checks["total"]

print("PASS: Real model-ready data verified")
print("Model-ready columns:", len(model_ready.columns))
print("Model-ready rows:", model_ready.count())
display(model_ready.limit(10))
